# **Data Preparation**:-

## Importing dataset:

In [1]:
import pandas as pd
df = pd.read_csv("diabetes_prediction_dataset.csv")
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


## Handle missing values:

In [2]:
missing_val = df.isnull().sum()
print("missing values:",missing_val)

missing values: gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


>no missing values

## See duplicates:

In [3]:
duplicates = df.duplicated().sum()
print("the duplicates are:",duplicates)

the duplicates are: 3854


>the duplicates are: 3854

## Remove duplicates:

In [4]:
df = df.drop_duplicates().reset_index(drop=True)
print("Removed duplicate rows:", duplicates)
print("Dataset shape after removing duplicates:", df.shape)

Removed duplicate rows: 3854
Dataset shape after removing duplicates: (96146, 9)


### see duplicates:

In [5]:
remaining_duplicates = df.duplicated().sum()
print("Remaining duplicates:", remaining_duplicates)
assert remaining_duplicates == 0

Remaining duplicates: 0


>duplicates have been removed

## Feature Engineering:

### see columns

In [6]:
df.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'],
      dtype='str')

### see value counts:

In [7]:
df["gender"].value_counts()

gender
Female    56161
Male      39967
Other        18
Name: count, dtype: int64

### unique values:

In [8]:
df["gender"].unique()

<StringArray>
['Female', 'Male', 'Other']
Length: 3, dtype: str

In [9]:
df["heart_disease"].value_counts()

heart_disease
0    92223
1     3923
Name: count, dtype: int64

In [10]:
df["cardio_risk"] = df["hypertension"] + df["heart_disease"]

df["age_bmi"] = df["age"] * df["bmi"]

In [11]:
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,cardio_risk,age_bmi
0,Female,80.0,0,1,never,25.19,6.6,140,0,1,2015.20
1,Female,54.0,0,0,No Info,27.32,6.6,80,0,0,1475.28
2,Male,28.0,0,0,never,27.32,5.7,158,0,0,764.96
3,Female,36.0,0,0,current,23.45,5.0,155,0,0,844.20
4,Male,76.0,1,1,current,20.14,4.8,155,0,2,1530.64


## Feature Selection:

>From the features I already have, which ones should I keep for the model?

##### Correlation:

In [12]:
cor = df.corr(numeric_only=True)
print(cor["diabetes"].sort_values(ascending=False))

diabetes               1.000000
blood_glucose_level    0.424336
HbA1c_level            0.406408
age_bmi                0.321812
age                    0.264927
cardio_risk            0.245177
bmi                    0.214932
hypertension           0.195710
heart_disease          0.170711
Name: diabetes, dtype: float64


## SelectKBest:

In [13]:
from sklearn.feature_selection import SelectKBest, f_classif

In [14]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

In [15]:
X_numeric = df[
    [
        "age",
        "hypertension",
        "heart_disease",
        "bmi",
        "HbA1c_level",
        "blood_glucose_level"
    ]
]

selector = SelectKBest(
    score_func=f_classif,
    k=4
)

X_selected = selector.fit_transform(X_numeric, y)

In [16]:
selected_features = X_numeric.columns[selector.get_support()]

print(selected_features)

Index(['age', 'bmi', 'HbA1c_level', 'blood_glucose_level'], dtype='str')


## Split X and y:

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Scaling:

Numerical columns → StandardScaler

In [18]:
numerical_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

Categorical columns → OneHotEncoder:

In [19]:
categorical_features = [
    "gender",
    "smoking_history"
]

## use column transformer

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [21]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

***************************************

# **Model Development**:-

Random Forest 

XGBoost 

Support Vector Machine 

Gradient Boosting 

Decision Tree

# *Random Forest* :

In [22]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

In [23]:
model.fit(X_train_processed, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [24]:
y_pred = model.predict(X_test_processed)

## predicted:

In [25]:
print(y_pred)

[0 0 1 ... 0 0 0]


## Model Evaluation:

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    log_loss,
    matthews_corrcoef
)

In [27]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9681747269890796
Precision: 0.9439551641313051
Recall: 0.6850668216153399
F1 Score: 0.793939393939394
Confusion Matrix:
[[17439    70]
 [  542  1179]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     17509
           1       0.94      0.69      0.79      1721

    accuracy                           0.97     19230
   macro avg       0.96      0.84      0.89     19230
weighted avg       0.97      0.97      0.97     19230



## ROC-AUC:

In [28]:
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.9580210745107297


# *XGBOOST*:

In [29]:
from xgboost import XGBClassifier

In [30]:
model = XGBClassifier(random_state=42)

In [31]:
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

In [32]:
y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9695787831513261
Precision: 0.9500792393026941
Recall: 0.6966879721092388
F1 Score: 0.8038887026483406
ROC-AUC: 0.9766440196158437


# *Decision Tree*

In [33]:
from sklearn.tree import DecisionTreeClassifier

In [34]:
model = DecisionTreeClassifier(random_state=42)

In [35]:
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

In [36]:
y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9493499739989599
Precision: 0.707615341856587
Recall: 0.7396862289366647
F1: 0.7232954545454545
ROC-AUC: 0.8552594964940252


# *Support Vector*:

In [37]:
from sklearn.svm import SVC

In [38]:
model = SVC(probability=True)

In [39]:
from sklearn.svm import SVC

model = SVC(probability=True)

# 1. Train
model.fit(X_train_processed, y_train)

# 2. Predict
y_pred = model.predict(X_test_processed)

# 3. Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Accuracy: 0.960894435777431
Precision: 0.9736070381231672
Recall: 0.5787332945961651
F1 Score: 0.7259475218658892


# *Gradient Boosting*:

In [40]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(random_state=42)

# Train
model.fit(X_train_processed, y_train)

# Predict
y_pred = model.predict(X_test_processed)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# ROC-AUC
y_prob = model.predict_proba(X_test_processed)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9709828393135725
Precision: 0.9866108786610879
Recall: 0.6850668216153399
F1 Score: 0.808641975308642
ROC-AUC: 0.9789697099083001


# **Model Optimization**:-

### Hyperparameter Tuning 

### Cross Validation

In [41]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [3, 5, 10],
            "min_samples_split": [2, 5]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [100, 200],
            "max_depth": [5, 10]
        }
    ),

    "XGBoost": (
        XGBClassifier(random_state=42),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5]
        }
    ),

    "SVM": (
        SVC(probability=True),
        {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf"]
        }
    ),

    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [2, 3]
        }
    )
}

for name, (model, params) in param_grids.items():

    grid = GridSearchCV(
        model,
        params,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )

    grid.fit(X_train_processed, y_train)

    print("\n", name)
    print("Best Parameters:", grid.best_params_)
    print("Best CV F1:", grid.best_score_)


 Decision Tree
Best Parameters: {'max_depth': 10, 'min_samples_split': 2}
Best CV F1: 0.8022484050803985

 Random Forest
Best Parameters: {'max_depth': 10, 'n_estimators': 100}
Best CV F1: 0.8009344160627998

 XGBoost
Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best CV F1: 0.8060253486679205


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



 SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Best CV F1: 0.7584045539146873

 Gradient Boosting
Best Parameters: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}
Best CV F1: 0.8054880885657631


## Feature Importance Analysis:

In [42]:
models = {
    name: model
    for name, (model, params) in param_grids.items()
}

predictions = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    predictions[name] = y_pred
    
    print(f"{name} predictions:")
    print(y_pred[:10])
    print()

Decision Tree predictions:
[0 0 1 0 0 0 0 0 0 0]

Random Forest predictions:
[0 0 1 0 0 0 0 0 0 0]

XGBoost predictions:
[0 0 1 0 0 0 0 0 0 0]



c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM predictions:
[0 0 1 0 0 0 0 0 0 0]

Gradient Boosting predictions:
[0 0 1 0 0 0 0 0 0 0]



## metrics:

In [43]:
for name, model in models.items():
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Decision Tree
Accuracy: 0.9493499739989599
Precision: 0.707615341856587
Recall: 0.7396862289366647
F1: 0.7232954545454545
ROC-AUC: 0.8552594964940252

Random Forest
Accuracy: 0.9681747269890796
Precision: 0.9439551641313051
Recall: 0.6850668216153399
F1: 0.793939393939394
ROC-AUC: 0.9580210745107297

XGBoost
Accuracy: 0.9695787831513261
Precision: 0.9500792393026941
Recall: 0.6966879721092388
F1: 0.8038887026483406
ROC-AUC: 0.9766440196158437

SVM
Accuracy: 0.960894435777431
Precision: 0.9736070381231672
Recall: 0.5787332945961651
F1: 0.7259475218658892
ROC-AUC: 0.9332284792590606

Gradient Boosting
Accuracy: 0.9709828393135725
Precision: 0.9866108786610879
Recall: 0.6850668216153399
F1: 0.808641975308642
ROC-AUC: 0.9789697099083001


***************

# **Save the Best Model**:-

In [50]:
import joblib

best_model_name = max(
    models,
    key=lambda name: f1_score(
        y_test,
        models[name].predict(X_test_processed)
    )
)
best_model = models[best_model_name]

joblib.dump(
    {
        "model": best_model,
        "preprocessor": preprocessor
    },
    "best_diabetes_model.pkl"
)
print(f"Saved best model: {best_model_name}")

Saved best model: Gradient Boosting


***********************

# **Simple Prediction Interface:**

In [52]:
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

# Load model and preprocessing transformer
model_paths = [
    Path("best_diabetes_model.pkl"),
    Path("Task 03") / "best_diabetes_model.pkl"
]
model_path = next((path for path in model_paths if path.exists()), None)

if model_path is None:
    st.error("best_diabetes_model.pkl was not found. Run the Save the Best Model cell first.")
    st.stop()

artifact = joblib.load(model_path)
if isinstance(artifact, dict):
    model = artifact["model"]
    preprocessor = artifact["preprocessor"]
else:
    model = artifact
    preprocessor = globals().get("preprocessor")
    if preprocessor is None:
        st.error("The saved model does not contain its preprocessor. Run the Save the Best Model cell again.")
        st.stop()

st.title("Diabetes Prediction")
st.write("Enter patient information to predict diabetes.")

# Inputs
gender = st.selectbox(
    "Gender",
    ["Female", "Male", "Other"]
)

age = st.number_input(
    "Age",
    min_value=0.0,
    max_value=100.0,
    value=30.0
)

hypertension = st.selectbox(
    "Hypertension",
    [0, 1]
)

heart_disease = st.selectbox(
    "Heart Disease",
    [0, 1]
)

smoking_history = st.selectbox(
    "Smoking History",
    ["never", "former", "current", "not current", "ever", "No Info"]
)

bmi = st.number_input(
    "BMI",
    min_value=0.0,
    max_value=100.0,
    value=25.0
)

hba1c = st.number_input(
    "HbA1c Level",
    min_value=0.0,
    max_value=20.0,
    value=5.5
)

blood_glucose = st.number_input(
    "Blood Glucose Level",
    min_value=0.0,
    max_value=500.0,
    value=100.0
)

# Prediction
if st.button("Predict"):

    input_data = pd.DataFrame({
        "gender": [gender],
        "age": [age],
        "hypertension": [hypertension],
        "heart_disease": [heart_disease],
        "smoking_history": [smoking_history],
        "bmi": [bmi],
        "HbA1c_level": [hba1c],
        "blood_glucose_level": [blood_glucose]
    })

    input_data_processed = preprocessor.transform(input_data)
    prediction = model.predict(input_data_processed)[0]

    if prediction == 1:
        st.error("Prediction: Diabetes")
    else:
        st.success("Prediction: No Diabetes")

2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 18:34:49.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar